# A Chatbot that Remembers, using LangGraph

In [ ]:
!pip install langgraph langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.5 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq

groq_api_key = userdata.get("GROQ_API_KEY")
llm = ChatGroq(api_key= groq_api_key,
               model="llama-3.1-8b-instant", temperature=0)

In [ ]:
from typing import TypedDict, List

class ChatState(TypedDict):
    messages: List[str]     # the running conversation
    name: str               # remembered user name
    preferences: str        # remembered preferences

In [ ]:
def chatbot(state: ChatState) -> dict:
    user_msg = state['messages'][-1]

    # Retrieve memory
    name = state.get('name', '')
    prefs = state.get('preferences', '')

    # Simple memory extraction rules
    if 'my name is' in user_msg.lower():
        name = user_msg.lower().split('my name is')[-1].strip().title()

    if 'i like' in user_msg.lower():
        prefs = user_msg.lower().split('i like')[-1].strip()

    # Context for LLM
    context = f"The user's name is {name}. They like {prefs}."

    # Generate response
    reply = llm.invoke(context + ' Reply to: ' + user_msg).content

    # Update state
    return {
        'messages': state['messages'] + [reply],
        'name': name,
        'preferences': prefs
    }

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ChatState)

builder.add_node('chatbot', chatbot)
builder.add_edge(START, 'chatbot')
builder.add_edge('chatbot', END)

graph = builder.compile()

In [ ]:
state = {
    'messages': [],
    'name': '',
    'preferences': ''
}

print("🤖 Chatbot started! Type 'quit' to exit.\n")

while True:
    user = input("You: ")

    if user.lower() == 'quit':
        print("👋 Goodbye!")
        break

    state['messages'].append(user)

    state = graph.invoke(state)

    print("Bot:", state['messages'][-1])

🤖 Chatbot started! Type 'quit' to exit.

You: My name is Arun
Bot: Nice to meet you, Arun. How's your day going so far?
You: What is my name?
Bot: Hello Arun, nice to meet you.
You: quit
👋 Goodbye!
